In [ ]:
import os
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# Definir los directorios de origen y destino
src_folder = "spectrums"
dst_folder = "spectrums training 100k"

# Crear la carpeta de destino si no existe
if not os.path.exists(dst_folder):
    os.makedirs(dst_folder)

# Listar todos los archivos .fits en el directorio de origen
fits_files = [f for f in os.listdir(src_folder) if f.lower().endswith(".fits")]

# Seleccionar 100000 archivos aleatorios
selected_files = random.sample(fits_files, 100000)

# Definir los rangos de redshift deseados
bin_ranges = [(0, 0.1), (0.1, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8)]

# Diccionario para llevar el conteo en cada rango
bin_counts = {br: 0 for br in bin_ranges}

# Diccionario con mínimos distintos para cada rango
# 0-1 hay > 100000
# 1-2 hay > 100000
# 2-3 hay > 100000
# 3-4 hay 35894
# 4-5 hay 2529
# 5-6 hay 900
# 6-7 hay 854
# 7-8 hay 245
min_required_values = {
    (0, 0.1): 20000,
    (0.1, 1): 15000,
    (1, 2): 20000,
    (2, 3): 15000,
    (3, 4): 5000,
    (4, 5): 2000,
    (5, 6): 900,
    (6, 7): 850,
    (7, 8): 240
}

# Lista para almacenar los valores de redshift y usarlos en el histograma
redshift_values = []

# Copiar archivos y extraer el valor de redshift de cada uno
for file in selected_files:
    src_path = os.path.join(src_folder, file)
    dst_path = os.path.join(dst_folder, file)
    
    # Verificar si el archivo existe, de lo contrario saltarlo
    if not os.path.exists(src_path):
        print(f"Archivo no encontrado: {file}. Saltando.")
        continue
    
    # Copiar el archivo
    shutil.copy(src_path, dst_path)
    print(f"Copiado: {file}")
    
    # Extraer el valor de redshift
    try:
        with fits.open(src_path) as hdul:
            redshift = hdul[2].data["Z"][0]
            redshift_values.append(redshift)
            # Determinar en qué rango cae el redshift y actualizar el conteo
            for br in bin_ranges:
                lower, upper = br
                if lower <= redshift < upper:
                    bin_counts[br] += 1
                    break
    except Exception as e:
        print(f"Error procesando {file}: {e}")

# Imprimir un reporte individual para cada rango con su mínimo respectivo
print("\nConteo por rangos de redshift:")
for br in bin_ranges:
    lower, upper = br
    count = bin_counts[br]
    min_required = min_required_values.get(br, 0)
    if count >= min_required:
        print(f"Rango {lower} - {upper}: {count} archivos (cumple con el mínimo de {min_required})")
    else:
        print(f"Rango {lower} - {upper}: {count} archivos (NO cumple con el mínimo de {min_required})")

Copiado: spec-8403-57834-0685.fits
Copiado: spec-0514-51994-0290.fits
Copiado: spec-4781-55653-0291.fits
Copiado: spec-6807-56429-0705.fits
Copiado: spec-2161-53878-0527.fits
Copiado: spec-0284-51943-0299.fits
Copiado: spec-1106-52912-0058.fits
Copiado: spec-6172-56269-0578.fits
Copiado: spec-4539-55865-0153.fits
Copiado: spec-1323-52797-0235.fits
Copiado: spec-8532-58022-0111.fits
Copiado: spec-1568-53169-0561.fits
Copiado: spec-1420-53146-0037.fits
Copiado: spec-6417-56308-0670.fits
Copiado: spec-5403-55979-0953.fits
Copiado: spec-2765-54535-0211.fits
Copiado: spec-7633-56931-0273.fits
Copiado: spec-10911-58256-0894.fits
Copiado: spec-5857-56092-0342.fits
Copiado: spec-8278-56990-0215.fits
Copiado: spec-5150-56329-0759.fits
Copiado: spec-10749-58485-0314.fits
Copiado: spec-1646-53498-0171.fits
Copiado: spec-4875-55677-0828.fits
Copiado: spec-9585-57780-0033.fits
Copiado: spec-0951-52398-0173.fits
Copiado: spec-3693-55208-0511.fits
Copiado: spec-2571-54055-0325.fits
Copiado: spec-1427

In [ ]:
# Definir los bordes de los bins para el histograma usando los rangos definidos
bin_edges = [br[0] for br in bin_ranges] + [bin_ranges[-1][1]]

# Crear el histograma
plt.figure(figsize=(18,16))
n, bins_hist, patches = plt.hist(redshift_values, bins=bin_edges, color='skyblue', edgecolor='black')
plt.xlabel("Redshift")
plt.ylabel("Número de archivos")
plt.title("Histograma de redshifts (bins definidos manualmente)")

# Dibujar líneas horizontales de referencia para cada rango con el mínimo requerido
for br in bin_ranges:
    lower, upper = br
    min_required = min_required_values.get(br, 0)
    # Dibujar una línea horizontal que cubre el rango del bin
    plt.hlines(y=min_required, xmin=lower, xmax=upper, colors='red', linestyles='dashed', linewidth=2)
    # Anotar el valor mínimo en el centro del rango
    x_center = (lower + upper) / 2
    plt.text(x_center, min_required + 1, f"min: {min_required}", color='red', ha='center', va='bottom')

plt.xlim(0, 7.1)
plt.show()